# Multi-Crop Plant Disease Training (Colab)
Run this notebook from top to bottom.

Expected dataset structure under `DATASET_ROOT`:
- Apple/Train, Apple/Val, Apple/Test
- Bell Pepper/Train, Bell Pepper/Val, Bell Pepper/Test
- Cherry/Train, Cherry/Val, Cherry/Test
- Corn (Maize)/Train, Corn (Maize)/Val, Corn (Maize)/Test
- Grape/Train, Grape/Val, Grape/Test
- Peach/Train, Peach/Val, Peach/Test
- Potato/Train, Potato/Val, Potato/Test
- Strawberry/Train, Strawberry/Val, Strawberry/Test
- Tomato/Train, Tomato/Val, Tomato/Test (optional but recommended)

In [ ]:
import importlib
drive_module = importlib.import_module('google.colab.drive')
drive_module.mount('/content/drive')

In [ ]:
# Update these paths if needed
PROJECT_ROOT = '/content/drive/MyDrive/Real_Project'
DATASET_ROOT = '/content/drive/MyDrive/plant_disease_dataset'

import os
if not os.path.exists(PROJECT_ROOT):
    raise FileNotFoundError(f'Project not found: {PROJECT_ROOT}')
if not os.path.exists(DATASET_ROOT):
    raise FileNotFoundError(f'Dataset not found: {DATASET_ROOT}')

%cd {PROJECT_ROOT}
print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATASET_ROOT:', DATASET_ROOT)

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt

In [ ]:
from pathlib import Path

required_splits = {'train', 'val', 'test'}
dataset_root = Path(DATASET_ROOT)

def has_required_splits(crop_dir: Path) -> bool:
    names = {p.name.lower() for p in crop_dir.iterdir() if p.is_dir()}
    return required_splits.issubset(names)

crop_dirs = sorted(
    [p for p in dataset_root.iterdir() if p.is_dir() and has_required_splits(p)],
    key=lambda p: p.name.lower()
)

print(f'Found {len(crop_dirs)} crop folders:')
for c in crop_dirs:
    print('-', c.name)

if not crop_dirs:
    raise ValueError('No crop folders with Train/Val/Test found. Check DATASET_ROOT.')
import re
import subprocess
import sys

EPOCHS = 15
IMG_SIZE = 224
BATCH_SIZE = 32
SKIP_EXISTING = True

OUTPUT_DIR = Path('model/saved_model')
ARTIFACTS_DIR = Path('model/artifacts')
TRAIN_SCRIPT = Path('model/src/train.py')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

if not TRAIN_SCRIPT.exists():
    raise FileNotFoundError(f'train.py not found at {TRAIN_SCRIPT}')

def slugify(name: str) -> str:
    slug = re.sub(r'[^a-zA-Z0-9]+', '_', name.strip().lower())
    return slug.strip('_')

python_exe = sys.executable
failed = []

In [ ]:
for crop in crop_dirs:
    crop_slug = slugify(crop.name)
    model_out = OUTPUT_DIR / f'plant_disease_{crop_slug}.h5'
    history_out = ARTIFACTS_DIR / f'training_history_{crop_slug}.json'
    metrics_out = ARTIFACTS_DIR / f'test_metrics_{crop_slug}.json'
    class_names_out = ARTIFACTS_DIR / f'class_names_{crop_slug}.json'

    if SKIP_EXISTING and model_out.exists():
        print(f'\n[SKIP] {crop.name}: model already exists at {model_out}')
        continue

    cmd = [
        python_exe, str(TRAIN_SCRIPT),
        '--data_root', str(crop),
        '--epochs', str(EPOCHS),
        '--img_size', str(IMG_SIZE),
        '--batch_size', str(BATCH_SIZE),
        '--model_out', str(model_out),
        '--history_out', str(history_out),
        '--metrics_out', str(metrics_out),
        '--class_names_out', str(class_names_out),
    ]

    print('\n' + '=' * 72)
    print(f'Training crop: {crop.name}')
    print('=' * 72)
    print(' '.join(cmd))

    code = subprocess.run(cmd).returncode
    if code != 0:
        failed.append(crop.name)
        print(f'[FAILED] {crop.name}')
    else:
        print(f'[DONE] {crop.name}')

In [ ]:
print('\n' + '=' * 72)
if failed:
    print('Training finished with failures:')
    for crop_name in failed:
        print('-', crop_name)
else:
    print('Training finished successfully for all crops.')

print('\nSaved models:')
for p in sorted(OUTPUT_DIR.glob('plant_disease_*.h5')):
    print('-', p)